In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MultiLabelBinarizer
import pandas as pd
import numpy as np

In [ ]:
class drop_columns(BaseEstimator, TransformerMixin):

  def __init__(self,columns_to_drop):
    self.columns_to_drop = columns_to_drop

  def fit(self, X, y=None):
    return self

  def transform(self,X):
    return X.drop(columns=self.columns_to_drop)

In [ ]:
class missingcategory(BaseEstimator, TransformerMixin):

  def __init__(self, columns_to_fill,missing_label):
    self.columns_to_fill = columns_to_fill
    self.missing_label = missing_label

  def fit(self, X, y=None):
    return self

  def transform(self, X):
    X[self.columns_to_fill] = X[self.columns_to_fill].fillna(self.missing_label)

    return X

In [ ]:
class renamer(BaseEstimator, TransformerMixin):

  def fit(self, X, y=None):
    self.rename_dict = {}

    for col in X.columns:
      org_col = col
      while "__" in col:
        col = col.split("__")[-1]

      self.rename_dict[org_col] = col

    return self

  def transform(self, X):
    X = X.copy()
    X = X.rename(columns=self.rename_dict)
    return X

In [ ]:
class AddMissingFlag(BaseEstimator, TransformerMixin):

    def __init__(self, column):
        self.column = column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X[f"{self.column}_missing"] = X[self.column].isna().astype(int)
        return X


In [ ]:
class gmsc_outlier_handle(BaseEstimator, TransformerMixin):
  def __init__(self,columns):
    self.columns = columns

  def fit(self, X, y=None):
    self.upper = {}
    for col in self.columns:
      if col =='RUUL':
        self.upper[col] = X[col].quantile(0.99)
      if col == 'DebtRatio':
        self.upper[col] = 273
    return self

  def transform(self, X):
    X = X.copy()
    for col in self.columns:
      X[col] = X[col].clip(upper=self.upper[col])
    return X

In [ ]:
class MultiLabelBinarizerCustom(BaseEstimator, TransformerMixin):

    def __init__(self,loan_types):
        self.loan_types = loan_types
        self.mlb = MultiLabelBinarizer(classes=loan_types)

    def fit(self, X, y=None):
        X.iloc[:,0] = X.iloc[:,0].str.split(',')
        self.mlb.fit(X)
        return self

    def transform(self, df):
        X = df.copy()

        print(df.columns)

        mlb_array = pd.DataFrame(
            self.mlb.transform(X.iloc[:,0]),
            columns=self.mlb.classes_,
            index=X.index
        )

        return mlb_array

In [ ]:
class aus_outlier_handle(BaseEstimator, TransformerMixin):
  def __init__(self,columns):
    self.columns = columns

  def fit(self, X, y=None):
    self.upper = {}
    for col in self.columns:
      if col =='A14':
        self.upper[col] = X[col].quantile(0.99)
    return self

  def transform(self, X):
    X = X.copy()
    for col in self.columns:
      if col in self.upper:
        X[col] = X[col].clip(upper=self.upper[col])
    return X

In [ ]:
def save_results_to_excel(
    metrics_df,
    classification_report,
    dataset_name,
    model_name,
    params,
    comments=None,
    file_path="model_results.xlsx"
):

    # -----------------------------
    # Convert metrics to single row
    # -----------------------------
    metrics_row = metrics_df.T.reset_index(drop=True)

    # -----------------------------
    # Flatten classification report
    # -----------------------------
    class_df = pd.DataFrame(classification_report).T
    class_df = class_df.stack().to_frame().T
    class_df.columns = [f"class_{c[0]}_{c[1]}" for c in class_df.columns]

    # -----------------------------
    # Metadata
    # -----------------------------
    meta = pd.DataFrame([{
        "dataset_name": dataset_name,
        "model_name": model_name,
        "params": str(params),
        "comments": comments,
        "timestamp": datetime.now()
    }])

    # -----------------------------
    # Combine everything
    # -----------------------------
    final_row = pd.concat([meta, metrics_row, class_df], axis=1)

    # -----------------------------
    # Save / append to Excel
    # -----------------------------
    if os.path.exists(file_path):
        existing = pd.read_excel(file_path)
        final_row = pd.concat([existing, final_row], ignore_index=True)

    final_row.to_excel(file_path, index=False)

    print(f"Results saved to {file_path}")